# Module 3 — Third‑Party Library Risks in AI

This notebook demonstrates **ML supply chain security** using the supporting files in this module:

- `requirements_vulnerable.txt` — vulnerable dependency set
- `requirements_secure.txt` — patched / hardened dependency set
- `approval_policy.json` — dependency approval rules
- `organizational_policy.md` — ML supply chain security policy
- `security_gate_config.json` — CI/CD security gate configuration
- `sbom.json` — CycloneDX SBOM for an ML pipeline
- `requirements_tensorflow.txt`, `requirements_pytorch.txt`, `requirements_sklearn.txt` — stack‑specific examples

We’ll walk through:
1. Understanding third‑party risk in ML
2. Analyzing vulnerable requirements
3. Comparing secure vs. vulnerable stacks
4. Applying organizational approval policies
5. Enforcing CI/CD security gates
6. Using SBOMs for vulnerability management
7. Building a secure dependency workflow


## 1. Why third‑party library risk matters in ML

ML systems are **deeply dependent** on third‑party libraries:

- Core numerical stacks (NumPy, SciPy)
- ML frameworks (PyTorch, TensorFlow, scikit‑learn)
- Data processing (Pillow, OpenCV, pandas)
- Networking and utilities (requests, urllib3)

This creates several risk categories:

- **Known CVEs** in popular libraries (e.g., Pillow, urllib3, TensorFlow)
- **Typosquatting / malicious packages** (e.g., fake `pytorch` variants)
- **License risk** (GPL/AGPL in commercial environments)
- **Unapproved or new publishers** with unknown reputation
- **Lack of SBOMs**, making incident response and zero‑day triage harder

This notebook turns those abstract risks into **concrete, repeatable checks**.

## 2. Load supporting files

We’ll load the module’s supporting files so we can reason about:

- Vulnerable vs. secure requirements
- Organizational approval policies
- CI/CD security gate configuration
- SBOM contents


In [ ]:
import json
from pathlib import Path

base = Path('.')

def read_text(path: Path) -> str:
    return path.read_text(encoding='utf-8') if path.exists() else ''

requirements_vulnerable = read_text(base / 'requirements_vulnerable.txt')
requirements_secure = read_text(base / 'requirements_secure.txt')
requirements_tf = read_text(base / 'requirements_tensorflow.txt')
requirements_torch = read_text(base / 'requirements_pytorch.txt')
requirements_sklearn = read_text(base / 'requirements_sklearn.txt')

approval_policy = json.loads(read_text(base / 'approval_policy.json') or '{}')
security_gate_config = json.loads(read_text(base / 'security_gate_config.json') or '{}')
sbom = json.loads(read_text(base / 'sbom.json') or '{}')

print('Loaded files:')
print('  requirements_vulnerable.txt:', bool(requirements_vulnerable))
print('  requirements_secure.txt   :', bool(requirements_secure))
print('  approval_policy.json      :', bool(approval_policy))
print('  security_gate_config.json :', bool(security_gate_config))
print('  sbom.json                 :', bool(sbom))

## 3. Parse vulnerable requirements

We’ll parse `requirements_vulnerable.txt` and extract the direct dependencies.

From the demo code:

> "Vulnerable requirements.txt with various issues"  
> "Contains: 12 direct dependencies, some with known vulnerabilities, older versions with CVEs"


In [ ]:
def parse_requirements(text: str):
    deps = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '==' in line:
            name, version = line.split('==', 1)
            deps.append((name.strip(), version.strip()))
    return deps

vuln_deps = parse_requirements(requirements_vulnerable)
secure_deps = parse_requirements(requirements_secure)

print('VULNERABLE DEPENDENCIES:')
for name, version in vuln_deps:
    print(f'  - {name}=={version}')

print('\nSECURE DEPENDENCIES:')
for name, version in secure_deps:
    print(f'  - {name}=={version}')

### 3.1 Known vulnerable packages (from the demo)

The demo script simulates `pip-audit` output and highlights specific CVEs:

> "Known vulnerabilities found: numpy 1.21.0, pillow 8.2.0, urllib3 1.26.4, tensorflow 2.5.0"  
> "Found 6 known vulnerabilities in 4 packages"

We’ll encode that knowledge as a small lookup table to illustrate how a scanner would flag them.

In [ ]:
import pandas as pd

known_vulns = [
    {"name": "numpy", "version": "1.21.0", "id": "GHSA-cfnr-3xfx-3fmv", "severity": "CRITICAL", "fix": "1.21.2"},
    {"name": "pillow", "version": "8.2.0", "id": "CVE-2021-34552", "severity": "HIGH", "fix": "8.3.0"},
    {"name": "pillow", "version": "8.2.0", "id": "CVE-2021-23437", "severity": "HIGH", "fix": "8.2.1"},
    {"name": "urllib3", "version": "1.26.4", "id": "CVE-2021-33503", "severity": "MEDIUM", "fix": "1.26.5"},
    {"name": "tensorflow", "version": "2.5.0", "id": "CVE-2021-37678", "severity": "HIGH", "fix": "2.5.1"},
    {"name": "tensorflow", "version": "2.5.0", "id": "CVE-2021-37679", "severity": "HIGH", "fix": "2.5.1"},
]

df_vulns = pd.DataFrame(known_vulns)
df_vulns

### 3.2 Match vulnerable requirements against known CVEs

This simulates what `pip-audit` / `safety` would do in CI/CD, but using the demo’s hard‑coded vulnerability list.

In [ ]:
vuln_index = {(r["name"], r["version"]): [] for r in known_vulns}
for r in known_vulns:
    vuln_index[(r["name"], r["version"])].append(r)

findings = []
for name, version in vuln_deps:
    key = (name, version)
    if key in vuln_index:
        for v in vuln_index[key]:
            findings.append({
                "package": name,
                "version": version,
                "cve": v["id"],
                "severity": v["severity"],
                "fix_version": v["fix"],
            })

df_findings = pd.DataFrame(findings)
df_findings

## 4. Typosquatting and malicious packages

From the demo:

> "SUSPICIOUS: Typosquatting attempts (these are fake examples)"  
> `pytorch-cuda-nightly`, `tensorflow-gpu-custom`, `scikit-learn-extra`

We’ll encode those as suspicious patterns and show how they would be flagged during review.

In [ ]:
suspicious_packages = [
    {
        "name": "pytorch-cuda-nightly",
        "issue": "Not an official PyTorch package",
        "correct": "torch (includes CUDA support)",
        "risk": "HIGH - Potential malware",
    },
    {
        "name": "tensorflow-gpu-custom",
        "issue": "Not an official TensorFlow package",
        "correct": "tensorflow or tensorflow-gpu",
        "risk": "HIGH - Potential malware",
    },
    {
        "name": "scikit-learn-extra",
        "issue": "Suspicious naming pattern",
        "correct": "scikit-learn",
        "risk": "MEDIUM - Verify legitimacy",
    },
]

pd.DataFrame(suspicious_packages)

## 5. License compliance and organizational policy

The **organizational policy** describes how ML supply chain security should work:

> "All production ML systems MUST use lock files"  
> "Hash verification REQUIRED"  
> "SBOM MUST be generated for every release"  
> "Vulnerability SLAs by Severity: CRITICAL 24h, HIGH 7 days, MEDIUM 30 days, LOW 90 days"  
> "All code changes MUST pass security gates (pip-audit + Safety, license compliance, dependency approval)"

We’ll also look at `approval_policy.json` and `security_gate_config.json` to see how this is enforced in code.

In [ ]:
import pprint

print('Approval policy rules:')
pprint.pp(approval_policy)

print('\nSecurity gate config:')
pprint.pp(security_gate_config)

### 5.1 Check vulnerable requirements against approval policy

From `approval_policy.json`:

- **ml-frameworks**: `torch`, `tensorflow`, `jax` → require `security-review`
- **restrictive-licenses**: `GPL-3.0`, `AGPL-3.0` → require `legal-review`
- **new-publishers**: packages younger than 90 days or low reputation → `security-review`
- **high-risk-categories**: `crypto`, `network`, `subprocess` → `security-review`

We’ll implement a simple check that flags ML frameworks for security review.

In [ ]:
ml_frameworks = set()
for rule in approval_policy.get("rules", []):
    if rule.get("category") == "ml-frameworks":
        ml_frameworks.update(rule.get("packages", []))

policy_findings = []
for name, version in vuln_deps:
    if name in ml_frameworks:
        policy_findings.append({
            "package": name,
            "version": version,
            "requires": "security-review",
            "reason": "Core ML framework with high impact",
        })

pd.DataFrame(policy_findings)

## 6. CI/CD security gates

From `security_gate_config.json`:

> "vulnerability_scanning: tools = [pip-audit, safety], fail_on = [CRITICAL, HIGH]"  
> "license_scanning: blocked_licenses = [GPL-3.0, AGPL-3.0], fail_on_unknown = true"  
> "dependency_approval: fail_on_unapproved = true"  
> "sbom_generation: required = true, format = CycloneDX"

We’ll simulate a **gate decision** based on the vulnerability findings above.

In [ ]:
fail_on = set(security_gate_config.get("vulnerability_scanning", {}).get("fail_on", []))

def gate_decision(df: pd.DataFrame) -> str:
    if df.empty:
        return "PASS (no known vulnerabilities)"
    severities = set(df["severity"].tolist())
    if severities & fail_on:
        return f"FAIL (found severities that must fail gate: {sorted(severities & fail_on)})"
    return f"WARN (vulnerabilities found, but below fail threshold: {sorted(severities)})"

decision = gate_decision(df_findings)
decision

## 7. Using SBOMs for ML supply chain visibility

From `sbom.json`:

> "SBOM MUST be generated for every release"  
> "Format: CycloneDX 1.4+"  
> "SBOMs MUST be queryable for vulnerability management"

We’ll inspect the SBOM and show how it can be used to answer:

- Which versions of NumPy / Torch / scikit‑learn are deployed?
- Which components would be affected by a new CVE?


In [ ]:
components = sbom.get("components", [])
df_sbom = pd.DataFrame([
    {
        "name": c.get("name"),
        "version": c.get("version"),
        "license": (c.get("licenses") or [{}])[0].get("license", {}).get("id"),
        "purl": c.get("purl"),
    }
    for c in components
])
df_sbom

### 7.1 Example: answering "Are we affected by CVE‑X?"

Given a new CVE for `numpy==1.24.3` or `torch==2.0.1`, we can query the SBOM instead of grepping code.


In [ ]:
def sbom_has(package: str, version: str | None = None) -> bool:
    for _, row in df_sbom.iterrows():
        if row["name"] == package and (version is None or row["version"] == version):
            return True
    return False

print('Has numpy 1.24.3? ', sbom_has('numpy', '1.24.3'))
print('Has torch 2.0.1?  ', sbom_has('torch', '2.0.1'))
print('Has sklearn 1.3.0?', sbom_has('scikit-learn', '1.3.0'))

## 8. Secure dependency workflow for ML

Based on the demo and policies, a **secure ML dependency workflow** looks like this:

1. **Design phase**
   - Choose libraries from an approved list
   - Avoid suspicious / typosquatted packages

2. **Development**
   - Pin exact versions (no `>=` ranges)
   - Use `requirements.in` + `pip-compile` for lock files
   - Run `pip-audit` / `safety` locally

3. **CI/CD security gates**
   - Run `pip-audit -r requirements.txt --strict`
   - Run `guarddog` for malicious package detection
   - Enforce `security_gate_config.json` (fail on CRITICAL/HIGH)

4. **Release**
   - Generate SBOM (CycloneDX)
   - Store SBOM in central repository

5. **Operations**
   - Daily CVE checks against SBOMs
   - Patch according to vulnerability SLAs
   - Quarterly review of approved packages


## 9. Summary — Third‑party library risks in AI

This notebook showed how third‑party risk in ML is **not just about CVEs**:

- Vulnerable versions of core libraries (NumPy, Pillow, urllib3, TensorFlow)
- Typosquatting and malicious packages in the ML ecosystem
- License and approval policies for ML frameworks
- CI/CD security gates that enforce vulnerability and license checks
- SBOMs as the backbone of ML supply chain visibility

The key takeaway:

> **Tools like `pip-audit`, `safety`, and SBOM generators are necessary, but not sufficient.**  
> They only become effective when combined with **clear policies**, **approval rules**, and **enforced security gates**.
